# TinyTurbo - GPU runs (Colab)

Runs the **existing** benchmark implementation on a real GPU. Nothing here is a separate
implementation: it drives `python -m tinyturbo` exactly as the local CPU workflow does.

Three steps, cheapest first:

| | config | scope | why |
|---|---|---|---|
| **A - smoke slice** | `configs/slice_gpu.yaml` | one prompt, 64 new tokens, 2 measured trials | proves the pipeline and the instrumentation work on *this* GPU |
| **B - main sweep** | `configs/gpu_main.yaml` | 10 prompts x 3 lengths, 3 measured trials each | the Stage 1 measurement of `docs/PLAN.md` |
| **0 + 4 - calibrate, profile** | `configs/gpu_main.yaml` | 3 primitives x (5 warm-ups + 20 timed passes), then 2 traced cases | measures the cost model and where the per-step time goes - section 10 |

**Rule 1:** if the preflight cell cannot find a CUDA device, stop. A CPU run would produce
numbers that look like results and are not.

**Rule 2:** each run downloads its own artefacts, keyed to the run id that execution
printed. No cell guesses which file is newest.


In [ ]:
#@title 1. Preflight - confirm the device actually assigned to this session
import subprocess, sys, torch

print('torch           :', torch.__version__)
print('cuda available  :', torch.cuda.is_available())
assert torch.cuda.is_available(), (
    'No CUDA device. In Colab: Runtime > Change runtime type > Hardware accelerator, '
    'pick a GPU (T4 is the free-tier option), then re-run this cell. Do not continue on CPU.'
)

def version_tuple(text):
    return tuple(int(part) for part in text.split('+')[0].split('.')[:2])

assert version_tuple(torch.__version__) >= (2, 4), (
    'transformers 5.3.0 requires torch>=2.4, found ' + torch.__version__
)

# Whatever Colab assigns is what we measure. Nothing below assumes a T4: the model name,
# compute capability and memory are read from the device and printed before anything runs.
capability = torch.cuda.get_device_capability(0)
properties = torch.cuda.get_device_properties(0)
print()
print('assigned gpu    :', torch.cuda.get_device_name(0))
print('compute cap.    : %d.%d' % capability)
print('total memory    :', round(properties.total_memory / 1024 ** 2), 'MiB')
print('cuda runtime    :', torch.version.cuda)
print('cudnn           :', torch.backends.cudnn.version())
print()
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

# configs/slice_gpu.yaml asks for float16 + sdpa, which is valid on sm75 and everything
# above. bfloat16 would need sm80+, so it is not requested: that is a separate experiment.
if capability < (8, 0):
    print('sm%d.%d (pre-Ampere): float16 + sdpa as configured is the correct choice.' % capability)
else:
    print('sm%d.%d (Ampere or newer): float16 + sdpa still runs, and the fingerprint' % capability)
    print('records what was actually used. A bfloat16/FlashAttention comparison is a later experiment.')

In [ ]:
#@title 2. Upload and unpack the project
from google.colab import files
uploaded = files.upload()  # choose TinyTurbo.zip from the project root (scripts/make_colab_bundle.py)
print('uploaded:', list(uploaded))

In [ ]:
!unzip -q -o /content/TinyTurbo.zip -d /content
%cd /content/TinyTurbo
!ls

In [ ]:
#@title 3. Install - only what the runtime does not already provide
# requirements-colab.txt pins transformers (the version whose assisted-generation
# behaviour is pinned in the tests) and pytest. It deliberately does not pin numpy
# or torch: the image's CUDA build and its compiled numpy are the ones to keep.
%pip install -q -r requirements-colab.txt
%pip install -q -e . --no-deps
!python -m tinyturbo --help

In [ ]:
#@title 4. Health check - the same suite that passes on CPU (158 tests)
!python -m pytest -q 2>&1 | tail -5


## 5. Helpers - pin a run id, then collect that run's artefacts

`--trials` is a command-line override and not part of a run's identity, so re-running the
same config **resumes**: a trial is skipped when its `(prompt, length, method, trial)` key
is already in the raw file. Warm-ups are keyed the same way and are skipped too, so a resumed
session appends only what is missing and starts appending straight away - no warm-up round
runs before its first new trial.

These helpers exist for one reason. A download must be pinned to the run *this session*
produced, so the run id is read out of this execution's own output and then checked against
the manifest on disk. Globbing `results/raw/slice-gpu-*.jsonl` and taking the last one sorts
by *name*, which is not the same as newest, and with two runs in one session it can quietly
hand back the other run's record.

Two derived files are written to shared paths (`results/benchmark_results.csv` has the same
path for every config), so each run's copy is snapshotted under its own run id before the
next report can overwrite it.


In [ ]:
#@title 5. Helpers - pin a run id, then collect that run's artefacts
import json, pathlib, shutil, subprocess, sys

RESULTS = pathlib.Path('results')
CONTENT = pathlib.Path('/content')


def tinyturbo(*args):
    """Run the project's own CLI and show what it said."""
    command = [sys.executable, '-m', 'tinyturbo', *args]
    print('$', ' '.join(command))
    done = subprocess.run(command, text=True, capture_output=True)
    print(done.stdout)
    if done.stderr.strip():
        print(done.stderr)
    return done


def run_and_pin(config_path, *extra):
    """Run a config and return the run id THIS execution printed.

    Never a glob, never sorted(...)[-1]: the id comes from this run's own stdout, is checked
    against the file the CLI says it wrote, and that file's manifest must name the same run.
    Anything else and a download could carry the wrong run's numbers.
    """
    done = tinyturbo('run', '--config', config_path, *extra)
    assert done.returncode == 0, f'run failed ({done.returncode}): {config_path}'

    ids = [
        line.split(':', 1)[1].strip()
        for line in done.stdout.splitlines()
        if line.startswith('run id')
    ]
    assert len(ids) == 1, f'expected exactly one run id in this execution, got {ids}'
    run_id = ids[0]

    raw = RESULTS / 'raw' / f'{run_id}.jsonl'
    assert raw.is_file(), f'the CLI named {run_id} but {raw} does not exist'
    manifest = json.loads(raw.read_text(encoding='utf-8').splitlines()[0])
    assert manifest['record_type'] == 'run', 'the first line should be the run manifest'
    assert manifest['run_id'] == run_id, 'that file belongs to a different run'

    print('pinned run id  :', run_id)
    print('scope          :', manifest['run_scope']['note'])
    print('hardware       :', manifest['hardware']['gpu_name'], '/', manifest['resolved']['dtype'])
    return run_id


def verify_and_report(config_path, run_id):
    """Verify the records, then derive the table and the figure from them."""
    checked = tinyturbo('verify', '--config', config_path)
    (CONTENT / f'{run_id}-verify.txt').write_text(checked.stdout + checked.stderr, encoding='utf-8')
    print('verify exit    :', checked.returncode)
    assert checked.returncode == 0, 'verify failed - do NOT report any number from this run'

    reported = tinyturbo('report', '--config', config_path)
    assert reported.returncode == 0, 'report failed'

    csv_source = RESULTS / 'benchmark_results.csv'
    plot_source = pathlib.Path('plots') / f'speedup_{run_id}.png'
    assert csv_source.is_file(), f'no derived CSV at {csv_source}'
    assert plot_source.is_file(), f'no figure at {plot_source}'
    shutil.copy(csv_source, CONTENT / f'{run_id}-benchmark_results.csv')
    shutil.copy(plot_source, CONTENT / f'{run_id}-figure.png')

    manifest = json.loads(
        (RESULTS / 'raw' / f'{run_id}.jsonl').read_text(encoding='utf-8').splitlines()[0]
    )
    fingerprint = {
        key: manifest[key]
        for key in ('run_id', 'created_utc', 'hardware', 'resolved', 'model_facts',
                    'assistant_controls', 'run_scope', 'git', 'command')
    }
    (CONTENT / f'{run_id}-fingerprint.json').write_text(
        json.dumps(fingerprint, indent=2), encoding='utf-8'
    )
    print('hardware       :', json.dumps(manifest['hardware']))
    return fingerprint


def download_run(run_id):
    """Download every artefact of one run, from /content, named for its run id."""
    from google.colab import files
    from IPython.display import Image, display

    artefacts = [
        RESULTS / 'raw' / f'{run_id}.jsonl',
        CONTENT / f'{run_id}-verify.txt',
        CONTENT / f'{run_id}-fingerprint.json',
        CONTENT / f'{run_id}-benchmark_results.csv',
        CONTENT / f'{run_id}-figure.png',
    ]
    for artefact in artefacts:
        assert artefact.is_file(), f'missing artefact: {artefact}'
        print(f'{artefact.name:<58s} {artefact.stat().st_size:>9,} bytes')

    display(Image(str(CONTENT / f'{run_id}-figure.png')))
    for artefact in artefacts:
        files.download(str(artefact))


## 6. Run A - the smoke slice

`configs/slice_gpu.yaml`: one prompt, 64 new tokens, `--trials 2` against the configured 5.

**Expect this cell to resume, not to measure.** The bundle carries this run's raw file,
because the slice is committed evidence, so the run finds its own 4 warm-ups and 2 measured
trials per arm already recorded. It prints *wrote 0 measured trial(s)* and *skipped 8 trial(s)
already present*, writes nothing, and hands back the numbers measured in an **earlier session**
- not a measurement of this GPU. That is the resume path working, not a failure, and it still
exercises the chain end to end: models load, the CLI runs, `verify` passes, `report` derives,
the artefacts download.

To re-measure the slice on *this* GPU, pass `--force`: it deletes the raw file first and
measures all four trials again, about a minute. You then get a second independent session's
numbers for the same configuration - worth having, since it is the cheapest check that two
sessions agree - at the cost of a second artefact set carrying the **same run id**, and so the
same file names.

Safe to interrupt either way: re-run the cell and it continues where it stopped.


In [ ]:
#@title 6. Run A - the smoke slice (one prompt, known good)
SMOKE_CONFIG = 'configs/slice_gpu.yaml'
SMOKE_RUN_ID = run_and_pin(SMOKE_CONFIG, '--trials', '2')

# The slice is committed evidence, so this cell resumes rather than measures: expect
# "wrote 0 measured trial(s)" and "skipped 8 trial(s) already present", and the numbers in the
# next cell to be the earlier session's. Add '--force' to re-measure it on this GPU instead.


In [ ]:
#@title 7. Run A - verify, report, collect, download
SMOKE_FINGERPRINT = verify_and_report(SMOKE_CONFIG, SMOKE_RUN_ID)
download_run(SMOKE_RUN_ID)


## 8. Run B - the main sweep (`docs/PLAN.md` Stage 1)

`configs/gpu_main.yaml`: 10 prompts x 3 generation lengths (32, 128, 256) x 2 methods = 30
points, with 2 warm-ups and 3 measured trials per point. That is 300 trials in all: 120
warm-ups and 180 measured records, 60 of them per generation length.

The cell below runs the whole thing in **one session**, which is the point of doing it in one
go: three measured trials per point under one set of conditions, with the configured warm-ups
ahead of them. The 1-trial version of this grid measured 14 minutes 31 seconds from its own
manifest to its last trial on a Tesla T4 (Colab free, 2 vCPUs, the same card and driver as the
64-token slice) - three rounds over the grid at ~4.8 minutes each - so five rounds, plus the
one-time ~6.8 GB checkpoint download, is roughly **25 minutes**.

**Expect this cell to resume, not to measure.** The sweep has now been run, and this
repository commits the result (`results/raw/gpu-main-9e26c12945e6.jsonl`, 301 records, a
24-minute T4 session on 2026-10-14), so a bundle built from this checkout carries it and the
cell finds all 300 trial keys already recorded. It prints *wrote 0 measured trial(s)* and
*skipped 300 trial(s) already present*, writes nothing, and hands back the numbers measured in
the **recorded session** - not a measurement of this GPU. That is the resume path working,
not a failure, and it still exercises the chain end to end: models load, the CLI runs,
`verify` passes, `report` derives, the artefacts download.

To actually re-measure the grid, run the cell with `--force` - which deletes the run's raw
file and starts again, so budget the ~25 minutes above - or change `name:` in
`configs/gpu_main.yaml` so this machine's trials get a file of their own. Do not do both and
then keep only one: the run id does not encode the trial count, so two files with the same
name and different contents are exactly the ambiguity the naming exists to avoid.

Short on time? `--trials 1` measures a single trial per point in about 15 minutes and the rest
can be appended later - but read the warm-up note above first: the resumed trials are not
preceded by any warm-up round.

Three things worth knowing before it starts:

* **The resume key is `(prompt, generation length, method, trial index)`.** It carries the
  length because every prompt runs at three of them: without it, trial 0 at 32 tokens would
  mark trial 0 at 256 tokens as done.
* **Resume is keyed to the configuration *and* the hardware.** If a later session is handed a
  different GPU, driver, CUDA or torch version, the run refuses to append to the previous file
  and tells you to rename the run (`name:` in the config) instead. That is deliberate: results
  from two machines are never pooled into one average.
* **A disconnect loses nothing.** Every trial is written and flushed as it finishes, and the
  cells above can be re-run from the install step.


In [ ]:
#@title 8. Run B - the main sweep (3 measured trials per point)
SWEEP_CONFIG = 'configs/gpu_main.yaml'
SWEEP_RUN_ID = run_and_pin(SWEEP_CONFIG)

# Short on time? One trial per point instead of three (~15 minutes), topped up later:
#   SWEEP_RUN_ID = run_and_pin(SWEEP_CONFIG, '--trials', '1')
# ...and then, in a later session, re-run the cells above and call it without --trials. Only
# the missing trials are appended - and the warm-ups are not re-run, so the resumed trials
# start cold. One session for all three trials is the cleaner measurement.


In [ ]:
#@title 9. Run B - verify, report, collect, download
SWEEP_FINGERPRINT = verify_and_report(SWEEP_CONFIG, SWEEP_RUN_ID)
download_run(SWEEP_RUN_ID)


## 10. Stage 0 and Stage 4 - calibrate, profile, re-report

Two stages, one session, because both must run on the *same* card as the run they describe.

**Stage 0 (`calibrate`)** times the three primitives of the plan's cost model directly, with a
populated cache and the same dtype, attention backend and lookahead as the sweep:

| primitive | what it is |
|---|---|
| `tau_t` | one target forward pass over 1 query token |
| `tau_t_verify` | one target forward pass over gamma+1 = 6 query tokens |
| `tau_d` | one assistant forward pass over 1 query token |

Each is 5 discarded warm-ups then 20 timed passes, reported as a median with an IQR, and the
raw samples are kept in the artifact. `rho = tau_t_verify / tau_t` and
`kappa = gamma * tau_d / tau_t` follow from them.

**Stage 4 (`profile`)** traces two representative cases from the sweep - the shortest
generation length and the longest - with the target's and assistant's forward passes
labelled, so kernel time can be attributed to drafting versus verification. Only labelled
forward passes are attributed; whatever wall-clock time is left over is reported as
unattributed rather than split up by assumption. This is the one stage a free Colab
runtime has not finished - it was killed by the memory limit (exit 137) - so the cell below
reports a failure here and carries on. Pass `--trace-active 4` to record a bounded window of
the loop instead; the artifact then says it covers a window and leaves the whole-run
residual unset, because a window cannot carry one.

**Then `report` is run again.** With a calibration that matches this run, it adds the cost
model lines to figure 2, writes figure 3, and prints the predicted-vs-observed table with
signed residuals.

**If the session landed on a different driver or torch build than the recorded sweep**, the
report will say so in a WARNING and will not use the calibration - that guard is deliberate
and must not be worked around: scoring a T4 sweep with a measurement from another machine
would produce a table that looks like evidence and is not. The calibration artifact is still
valid evidence about *this* machine; it just cannot be applied to that run.

**You do not need to run the Run A or Run B cells first.** The sweep these stages score is
already committed in the bundle (301 records, `gpu-main-9e26c12945e6`), and this cell pins
itself to that raw file by name and manifest. Cells 6-9 above are only needed if you want a
second session's numbers for the same points.


In [ ]:
#@title 10. Stage 0 + Stage 4 - calibrate, profile, re-report, download
import json, pathlib, shutil

# This cell is self-contained: it pins itself to the sweep already committed in this bundle,
# so the Run A and Run B cells above do NOT need to be executed first. The run id is taken
# from the raw file's own name and then checked against the manifest inside it - never from a
# glob order, which is how two runs in one session get mixed up.
SWEEP_CONFIG = 'configs/gpu_main.yaml'
_sweeps = sorted(RESULTS.glob('raw/gpu-main-*.jsonl'))
assert len(_sweeps) == 1, f'expected exactly one committed gpu-main raw file, found {_sweeps}'
SWEEP_RUN_ID = _sweeps[0].stem
_manifest = json.loads(_sweeps[0].read_text(encoding='utf-8').splitlines()[0])
assert _manifest['run_id'] == SWEEP_RUN_ID, 'that file belongs to a different run'
print('scoring        :', SWEEP_RUN_ID)
print('scope          :', _manifest['run_scope']['note'])
print('hardware       :', _manifest['hardware']['gpu_name'], '/', _manifest['resolved']['dtype'])
print()

# Stage 0: the cost-model primitives, measured on this card with the sweep's settings.
calibrated = tinyturbo('calibrate', '--config', SWEEP_CONFIG)
assert calibrated.returncode == 0, 'calibration failed'
CALIBRATION = RESULTS / 'calibration' / f'{SWEEP_RUN_ID}-calibration.json'
assert CALIBRATION.is_file(), f'no calibration artifact at {CALIBRATION}'

# Stage 4: two representative cases, traced with labelled forward passes. OPTIONAL - a
# free Colab session has not finished this (exit 137, the memory limit), so a failure is
# reported and skipped rather than allowed to abort the cell. Add '--trace-active', '4'
# to the call below to record a bounded window of the loop instead of the whole thing.
profiled = tinyturbo('profile', '--config', SWEEP_CONFIG)
if profiled.returncode != 0:
    print('Stage 4 did not produce an artifact - expected on a free runtime, see above.')
PROFILING = RESULTS / 'profiling' / f'{SWEEP_RUN_ID}-profiling.json'
print('Stage 4 artifact written      :', PROFILING.is_file())

# Re-report: this is where the cost-model lines, figure 3 and the predicted-vs-observed
# table appear - but only if the calibration describes this run.
reported = tinyturbo('report', '--config', SWEEP_CONFIG)
assert reported.returncode == 0, 'report failed'
MODEL_APPLIED = 'rho=' in reported.stdout
print('cost model applied to the sweep :', MODEL_APPLIED)
if not MODEL_APPLIED:
    print('NOTE: read the WARNING lines above - the calibration was not used, because it does')
    print('      not describe this run. Do not present its predicted-vs-observed table as if')
    print('      it did. The artifact is still a valid measurement of THIS machine.')

artefacts = [
    (CALIBRATION, f'{SWEEP_RUN_ID}-calibration.json'),
    (PROFILING, f'{SWEEP_RUN_ID}-profiling.json'),
    (RESULTS / 'benchmark_results.csv', f'{SWEEP_RUN_ID}-benchmark_results.csv'),
    (pathlib.Path('plots') / f'accepted_len_vs_speedup_{SWEEP_RUN_ID}.png', f'{SWEEP_RUN_ID}-figure2.png'),
    (pathlib.Path('plots') / f'time_attribution_{SWEEP_RUN_ID}.png', f'{SWEEP_RUN_ID}-figure3.png'),
]
collected = []
for source, name in artefacts:
    if not source.is_file():
        print(f'missing (skipped): {source}')
        continue
    target = CONTENT / name
    shutil.copy(source, target)
    collected.append(target)
    print(f'{name:<52s} {target.stat().st_size:>9,} bytes')

from google.colab import files
from IPython.display import Image, display
for target in collected:
    if target.suffix == '.png':
        display(Image(str(target)))
for target in collected:
    files.download(str(target))


## What these runs do and do not establish

**Run A establishes** that the pipeline works on a GPU, that the forward-pass instrumentation
works there, and that assisted greedy is token-identical to plain greedy on this hardware.
One prompt and two trials is a validity check, not a result.

**Run B establishes** the Stage 1 evidence of `docs/PLAN.md`: tokens accepted per target
forward pass at three generation lengths across ten prompts (H4), how much prefill dilutes the
end-to-end number as generation length grows (H5), and whether exactness holds everywhere
rather than twice (H6). It does **not** settle the lookahead question (Stage 2), the cost model
(Stage 0 and Stage 4), or anything about sm80+ hardware (H8) - those are separate runs with
their own configurations.

**Four things to check in the output:**

1. `tokens per target forward pass` - exactly **1.00** on the baseline arm. If it is not, the
   instrumentation is wrong and no other number here means anything.
2. `output identical to baseline: True` - assisted *greedy* is lossless, so a mismatch is a
   bug to investigate, not a finding.
3. The `hardware` line must name a GPU. If it says CPU, the run is not a measurement.
4. For Run B, the summary must show one line per method **and length** - `assisted @ 32`,
   `assisted @ 128`, `assisted @ 256`. A single `assisted` line means the file holds one
   length only.

**After the session**, on the machine that owns this repository:

```text
results/raw/<run_id>.jsonl                      the evidence, straight into results/raw/
results/sessions/<run_id>/benchmark_results.csv the table as downloaded
results/sessions/<run_id>/verify_output.txt     <- rename <run_id>-verify.txt
results/sessions/<run_id>/environment_fingerprint.json
plots/speedup_<run_id>.png                      the figure
```

Then `python -m tinyturbo report --config configs/gpu_main.yaml` regenerates the table and the
figure from the raw records alone, and `results/README.md` gets a row for the new run: its id,
its hardware, the command, and what it is and is not. Raw records are never edited.
